# SC4020 Project 2 — Task 1: Analysis of Drug–Target Datasets

## Step 1: Load the data and check it

This notebook expects a `data/` folder next to it, containing `davis/` and `kiba/` from the DeepDTA repository.

In [ ]:
# Run once to install the libraries (skip if already installed)
%pip install pandas numpy matplotlib seaborn

In [ ]:
import json
import pickle
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")

### Load ligands, targets and the score matrix

`ligands_can.txt` and `proteins.txt` are JSON dictionaries (ID → SMILES / sequence). `Y` is a pickled matrix whose rows follow the ligand order and columns follow the target order, so the dictionary order must be kept.

In [ ]:
def load_dataset(name):
    folder = DATA_DIR / name
    with open(folder / "ligands_can.txt") as f:
        ligands = json.load(f, object_pairs_hook=OrderedDict)
    with open(folder / "proteins.txt") as f:
        proteins = json.load(f, object_pairs_hook=OrderedDict)
    with open(folder / "Y", "rb") as f:
        Y = np.asarray(pickle.load(f, encoding="latin1"), dtype=float)  # file was saved with Python 2
    return ligands, proteins, Y

davis_ligands, davis_proteins, davis_Y = load_dataset("davis")
kiba_ligands, kiba_proteins, kiba_Y = load_dataset("kiba")

### Sanity checks against the project brief

Expected: DAVIS 68 × 442 with 30,056 observed pairs; KIBA 2,111 × 229 with 118,254 observed pairs.

In [ ]:
def summarise(name, ligands, proteins, Y):
    observed = np.isfinite(Y).sum()
    return {
        "dataset": name,
        "ligands": len(ligands),
        "targets": len(proteins),
        "matrix shape": Y.shape,
        "observed pairs": int(observed),
        "missing pairs": int(Y.size - observed),
        "sparsity": round(1 - observed / Y.size, 4),
        "unique SMILES": len(set(ligands.values())),
        "unique sequences": len(set(proteins.values())),
        "min score": np.nanmin(Y),
        "max score": np.nanmax(Y),
    }

pd.DataFrame([
    summarise("DAVIS (Kd, nM)", davis_ligands, davis_proteins, davis_Y),
    summarise("KIBA", kiba_ligands, kiba_proteins, kiba_Y),
]).set_index("dataset").T

### Example records (long format)

One row per observed ligand–target pair, which is the format the later tasks will use.

In [ ]:
def to_long(ligands, proteins, Y):
    lig_ids, lig_smiles = list(ligands.keys()), list(ligands.values())
    tgt_ids, tgt_seqs = list(proteins.keys()), list(proteins.values())
    rows, cols = np.where(np.isfinite(Y))
    return pd.DataFrame({
        "ligand_id": [lig_ids[i] for i in rows],
        "target_id": [tgt_ids[j] for j in cols],
        "smiles": [lig_smiles[i] for i in rows],
        "sequence": [tgt_seqs[j] for j in cols],
        "score": Y[rows, cols],
    })

davis_df = to_long(davis_ligands, davis_proteins, davis_Y)
kiba_df = to_long(kiba_ligands, kiba_proteins, kiba_Y)

def preview(df, n=5):
    # shorten long strings so the table is readable
    out = df.head(n).copy()
    out["smiles"] = out["smiles"].str.slice(0, 30) + "…"
    out["sequence"] = out["sequence"].str.slice(0, 30) + "…"
    return out

display(preview(davis_df))
display(preview(kiba_df))

## Step 2: Data cleaning

1. **Validity:** drop non-finite scores; DAVIS also requires $K_d > 0$ (needed for the log).
2. **Scale:** convert DAVIS to $\mathrm{p}K_d = 9 - \log_{10}(K_d)$ with $K_d$ in nM. KIBA scores are used as supplied (higher = stronger).
3. **Abnormal values are flagged, not removed:** DAVIS screening-limit records ($K_d = 10{,}000$ nM, i.e. $\mathrm{p}K_d = 5$) and KIBA extreme scores.
4. **Deduplication:** pairs with the same ligand SMILES and target sequence are merged and their scores averaged (in $\mathrm{p}K_d$ space for DAVIS).

In [ ]:
DAVIS_LIMIT_PKD = 5.0   # Kd = 10,000 nM screening limit

def validate(df, name):
    n0 = len(df)
    df = df[np.isfinite(df["score"])].copy()
    if name == "DAVIS":
        df = df[df["score"] > 0].copy()          # Kd must be positive
        df["kd_nM"] = df["score"]
        df["score"] = 9 - np.log10(df["kd_nM"])  # pKd
    print(f"{name}: {n0} records -> {len(df)} valid ({n0 - len(df)} removed)")
    return df

davis_valid = validate(davis_df, "DAVIS")
kiba_valid = validate(kiba_df, "KIBA")

### Abnormal values (kept, but flagged)

In [ ]:
davis_valid["at_screening_limit"] = np.isclose(davis_valid["score"], DAVIS_LIMIT_PKD)

q1, q3 = kiba_valid["score"].quantile([0.25, 0.75])
iqr = q3 - q1
kiba_valid["extreme_low"] = kiba_valid["score"] < q1 - 3 * iqr
kiba_valid["extreme_high"] = kiba_valid["score"] > q3 + 3 * iqr

abnormal = pd.DataFrame({
    "records": [
        davis_valid["at_screening_limit"].sum(),
        (kiba_valid["score"] == 0).sum(),
        kiba_valid["extreme_low"].sum(),
        kiba_valid["extreme_high"].sum(),
    ],
}, index=[
    "DAVIS at screening limit (pKd = 5)",
    "KIBA score exactly 0",
    f"KIBA extreme low (< Q1 - 3 IQR = {q1 - 3*iqr:.2f})",
    f"KIBA extreme high (> Q3 + 3 IQR = {q3 + 3*iqr:.2f})",
])
abnormal["% of dataset"] = [
    100 * abnormal["records"].iloc[0] / len(davis_valid),
    *(100 * abnormal["records"].iloc[1:] / len(kiba_valid)),
]
abnormal.round(2)

### Deduplication by SMILES + sequence

Some IDs share identical inputs: in DAVIS several mutant targets (e.g. `BRAF` and `BRAF(V600E)`) have the same supplied sequence, and in KIBA some ligand IDs share a canonical SMILES (stereoisomers). Since models will see SMILES and sequences, such pairs are indistinguishable and their scores are averaged. Each merged entity keeps a representative ID (the first in file order) and the full list of original IDs.

In [ ]:
def deduplicate(df, extra_flags=()):
    # one representative ID per unique SMILES / sequence
    lig = (df.groupby("smiles", sort=False)["ligand_id"]
             .agg(lambda s: list(dict.fromkeys(s))).rename("ligand_ids").reset_index())
    lig["ligand_key"] = lig["ligand_ids"].str[0]
    tgt = (df.groupby("sequence", sort=False)["target_id"]
             .agg(lambda s: list(dict.fromkeys(s))).rename("target_ids").reset_index())
    tgt["target_key"] = tgt["target_ids"].str[0]

    df = df.merge(lig[["smiles", "ligand_key"]], on="smiles").merge(tgt[["sequence", "target_key"]], on="sequence")
    agg = {"score": ["mean", "std", "min", "max", "count"]}
    agg.update({flag: "mean" for flag in extra_flags})   # fraction of merged records with the flag
    pairs = df.groupby(["ligand_key", "target_key"], sort=False).agg(agg)
    pairs.columns = ["score", "score_std", "score_min", "score_max", "n_merged", *[f"frac_{f}" for f in extra_flags]]
    pairs = pairs.reset_index()

    lig["ligand_ids"] = lig["ligand_ids"].str.join("|")
    tgt["target_ids"] = tgt["target_ids"].str.join("|")
    return pairs, lig, tgt

davis_pairs, davis_lig, davis_tgt = deduplicate(davis_valid, extra_flags=["at_screening_limit"])
kiba_pairs, kiba_lig, kiba_tgt = deduplicate(kiba_valid, extra_flags=["extreme_low", "extreme_high"])

### Original vs. deduplicated counts

In [ ]:
def counts(raw, pairs, lig, tgt):
    return {
        ("Ligands", "original IDs"): raw["ligand_id"].nunique(),
        ("Ligands", "deduplicated"): len(lig),
        ("Targets", "original IDs"): raw["target_id"].nunique(),
        ("Targets", "deduplicated"): len(tgt),
        ("Observed pairs", "original"): len(raw),
        ("Observed pairs", "deduplicated"): len(pairs),
        ("Observed pairs", "merged groups (n > 1)"): int((pairs["n_merged"] > 1).sum()),
        ("Matrix", "sparsity after dedup"): round(1 - len(pairs) / (len(lig) * len(tgt)), 4),
    }

count_table = pd.DataFrame({
    "DAVIS": counts(davis_df, davis_pairs, davis_lig, davis_tgt),
    "KIBA": counts(kiba_df, kiba_pairs, kiba_lig, kiba_tgt),
}, dtype=object)
count_table

### How much do merged duplicates disagree?

A large spread means the averaged score hides real differences (e.g. a mutant binding very differently from the wild type).

In [ ]:
def disagreement(pairs):
    m = pairs[pairs["n_merged"] > 1]
    spread = m["score_max"] - m["score_min"]
    return {
        "merged groups": len(m),
        "mean within-group std": round(m["score_std"].mean(), 3),
        "max spread (max - min)": round(spread.max(), 3),
        "groups with spread > 1": int((spread > 1).sum()),
    }

display(pd.DataFrame({"DAVIS (pKd)": disagreement(davis_pairs), "KIBA": disagreement(kiba_pairs)}))

# merged DAVIS target groups, e.g. wild type + mutants sharing one sequence
davis_tgt[davis_tgt["target_ids"].str.contains("|", regex=False)][["target_key", "target_ids"]].head(10)

### Save the cleaned data for Tasks 2 and 3

Pairs files hold `ligand_key, target_key, score`; the lookup files map each key to its SMILES or sequence and original IDs. Sequences are stored once per target to keep files small.

In [ ]:
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(exist_ok=True)

for name, pairs, lig, tgt in [("davis", davis_pairs, davis_lig, davis_tgt),
                              ("kiba", kiba_pairs, kiba_lig, kiba_tgt)]:
    pairs.to_csv(CLEAN_DIR / f"{name}_pairs.csv", index=False)
    lig[["ligand_key", "ligand_ids", "smiles"]].to_csv(CLEAN_DIR / f"{name}_ligands.csv", index=False)
    tgt[["target_key", "target_ids", "sequence"]].to_csv(CLEAN_DIR / f"{name}_targets.csv", index=False)

sorted(p.name for p in CLEAN_DIR.iterdir())